# EnergyCast V1 — Step 7: Feature Engineering Pipeline
## Electricity Price Forecasting System (Indian Electricity Market)

**Objective:** Transform raw, synchronized power system, market, weather, and temporal variables into a high-dimensional, strictly leakage-free feature matrix for short-term hourly electricity price forecasting.  
This notebook documents the mathematical formulation and implementation of autoregressive price lags, rolling volatility regimes, Net Demand, cyclical calendar encodings, and meteorological features across 28,176 clean hourly records.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
FEATURES_PATH = Path('../data/processed/features/energycast_features_hourly_2022_2025.csv')
DICT_PATH = Path('../data/processed/features/energycast_feature_dictionary.csv')
print('Features CSV exists:', FEATURES_PATH.exists())
print('Dictionary CSV exists:', DICT_PATH.exists())

### 1. Load Processed Feature Matrix & Dictionary
We inspect the engineered dataset dimensions, timestamp range, and verify zero nulls.

In [ ]:
df = pd.read_csv(FEATURES_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
dict_df = pd.read_csv(DICT_PATH)

print(f'Feature Matrix Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Timeline: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Total Missing / Null Values: {df.isnull().sum().sum()}')
display(dict_df.head(10))

### 2. Autoregressive Lags & Dynamic Rolling Volatility Bands
Visualizing actual MCP ($y_t$), daily lag ($y_{t-24}$), and the 24-hour moving average with $\pm 1\sigma$ dynamic volatility bands.

In [ ]:
sample = df.iloc[500:850] # 14-day sample
plt.figure(figsize=(14, 5))
plt.plot(sample['timestamp'], sample['mcp_rs_per_mwh'], label='Actual MCP ($y_t$)', color='#1F4E79', lw=1.5)
plt.plot(sample['timestamp'], sample['mcp_lag_24'], label='Lag 24 ($y_{t-24}$)', color='#E67E22', ls=':', lw=1.2)
plt.plot(sample['timestamp'], sample['mcp_rolling_mean_24'], label='24h Rolling Mean', color='#27AE60', lw=1.8)

upper = sample['mcp_rolling_mean_24'] + sample['mcp_rolling_std_24']
lower = sample['mcp_rolling_mean_24'] - sample['mcp_rolling_std_24']
plt.fill_between(sample['timestamp'], lower, upper, color='#27AE60', alpha=0.15, label='$\pm 1\sigma$ Volatility Envelope')

plt.title('Autoregressive Price Tracking & Dynamic 24-Hour Volatility Envelope', fontweight='bold')
plt.ylabel('Price (Rs/MWh)')
plt.xlabel('Date')
plt.legend(loc='upper right')
plt.show()

### 3. Continuous Cyclical Calendar Encodings (Unit Circle)
Verifying that the discrete hour of the day (0 to 23) has been mapped smoothly to circular sine and cosine coordinates.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sub_hours = df.drop_duplicates(subset=['hour_sin', 'hour_cos'])
ax.scatter(sub_hours['hour_cos'], sub_hours['hour_sin'], color='#1F4E79', s=60, zorder=3)
circle = plt.Circle((0, 0), 1.0, color='#CCCCCC', fill=False, linestyle='--', lw=1.2)
ax.add_artist(circle)
ax.axhline(0, color='gray', lw=0.6)
ax.axvline(0, color='gray', lw=0.6)
ax.set_xlim(-1.25, 1.25)
ax.set_ylim(-1.25, 1.25)
ax.set_aspect('equal')
ax.set_title('24-Hour Continuous Cyclical Unit Circle Mapping', fontweight='bold')
ax.set_xlabel('hour_cos = $\cos(2\pi \cdot \mathrm{hour} / 24)$')
ax.set_ylabel('hour_sin = $\sin(2\pi \cdot \mathrm{hour} / 24)$')
plt.show()

### 4. Feature Correlation Ranking Against Target (MCP)
Evaluating Spearman rank correlations across all 59 engineered predictors against ground-truth price.

In [ ]:
num_cols = [c for c in df.columns if c != 'timestamp']
corr_series = df[num_cols].corr(method='spearman')['mcp_rs_per_mwh'].drop('mcp_rs_per_mwh')
top_20 = corr_series.abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
top_20_signed = corr_series.loc[top_20.index].sort_values(ascending=True)
colors = ['#D9534F' if v < 0 else '#1F4E79' for v in top_20_signed.values]
plt.barh(top_20_signed.index, top_20_signed.values, color=colors, alpha=0.85)
plt.axvline(0, color='gray', lw=0.8)
plt.title('Top 20 Engineered Features by Spearman Rank Correlation (\rho)', fontweight='bold')
plt.xlabel('Spearman Correlation with MCP')
plt.show()

### 5. Summary & Hand-off to Step 8
With 60 validated columns, strict leakage prevention, and zero null values across 28,176 hourly observations, the feature matrix is ready for **Step 8: Chronological Train / Validation / Test Splitting**.